# 第七讲基础习题参考答案

本 notebook 提供 `07c_exercises.ipynb` 中 8 道基础巩固习题的完整参考实现与输出说明。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
torch.manual_seed(42)
np.random.seed(42)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 第一部分：PyTorch 张量、自动微分与多层感知机

### 习题 1：张量行标准化与矩阵乘法

In [ ]:
X = torch.tensor([
    [300.0, 1.5, 12.0],
    [450.0, 2.1, 18.0],
    [600.0, 3.0, 24.0],
    [750.0, 3.6, 30.0],
], dtype=torch.float32)
w = torch.tensor([[0.8], [-0.5], [1.2]], dtype=torch.float32)

col_mean = X.mean(dim=0)
col_std = X.std(dim=0)
X_norm = (X - col_mean) / col_std
y_lin = X_norm @ w

print("Column Mean:", col_mean)
print("Column Std: ", col_std)
print("Normalized X:\n", X_norm)
print("Linear output y_lin:\n", y_lin)

---
### 习题 2：用 Autograd 求多元自由能函数的偏导数

In [ ]:
T = torch.tensor(3.0, requires_grad=True)
c = torch.tensor(2.0, requires_grad=True)
G = 2 * (c ** 3) - 3 * T * (c ** 2) + (T ** 2)
G.backward()

print(f"G(3, 2) = {G.item():.1f}")
print(f"dG/dT = {T.grad.item():.1f} (手算解析值: -6.0)")
print(f"dG/dc = {c.grad.item():.1f} (手算解析值: -12.0)")

---
### 习题 3：手算与验证多层感知机的参数总数

In [ ]:
manual_count = (10 * 32 + 32) + (32 * 16 + 16) + (16 * 1 + 1)
net = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
pytorch_count = sum(p.numel() for p in net.parameters())
print("Manual count: ", manual_count)
print("PyTorch count:", pytorch_count)

---
### 习题 4：编写自定义 `nn.Module` 并用 Adam 拟合非线性曲线

In [ ]:
torch.manual_seed(0)
x_data = torch.linspace(-2.5, 2.5, 120).unsqueeze(1)
y_data = (torch.cos(2 * x_data) + 0.3 * x_data + 0.08 * torch.randn_like(x_data)).squeeze(1)

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = SimpleMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for epoch in range(200):
    pred = model(x_data)
    loss = loss_fn(pred, y_data)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Final MSE Loss after 200 epochs: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    y_fit = model(x_data).numpy()

fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.scatter(x_data.squeeze().numpy(), y_data.numpy(), s=18, alpha=0.6, label="Noisy Data")
ax.plot(x_data.squeeze().numpy(), y_fit, color="#d62728", lw=2.2, label="SimpleMLP Fit")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Nonlinear Curve Fitting with Custom nn.Module")
ax.legend()
plt.tight_layout()
plt.show()

---
### 习题 5：实现 MC Dropout 推理并比较内插与外推不确定性

In [ ]:
torch.manual_seed(42)
dropout_net = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(64, 1),
)
x_tr_mc = torch.linspace(-2.0, 2.0, 80).unsqueeze(1)
y_tr_mc = torch.sin(x_tr_mc)
opt_mc = torch.optim.Adam(dropout_net.parameters(), lr=1e-2)
for _ in range(150):
    dropout_net.train()
    loss = F.mse_loss(dropout_net(x_tr_mc), y_tr_mc)
    opt_mc.zero_grad(); loss.backward(); opt_mc.step()

x_probe = torch.tensor([[0.5], [5.0]])

dropout_net.train()  # 推理时保持 Dropout 开启
with torch.no_grad():
    mc_samples = torch.cat([dropout_net(x_probe).T for _ in range(100)], dim=0)  # (100, 2)

means = mc_samples.mean(dim=0).numpy()
stds = mc_samples.std(dim=0).numpy()
print(f"x = 0.5 (训练区内插): 预测均值 = {means[0]:+.3f}, MC Dropout std = {stds[0]:.3f}")
print(f"x = 5.0 (训练区外推): 预测均值 = {means[1]:+.3f}, MC Dropout std = {stds[1]:.3f} (外推不确定性显著升高)")

---
## 第二部分：二维离散卷积与金相微结构 CNN

### 习题 6：用水平 Sobel 核提取层片状组织的水平相界

In [ ]:
yy, xx = np.mgrid[0:32, 0:32].astype(np.float32)
horiz_np = ((np.sin(2 * np.pi * yy / 8.0) > 0).astype(np.float32) * 0.7 + 0.15)
horiz_img = torch.tensor(horiz_np)[None, None, :, :]

k_sobel_y = torch.tensor([[[[-1.0, -2.0, -1.0],
                            [ 0.0,  0.0,  0.0],
                            [ 1.0,  2.0,  1.0]]]])
edge_y = F.conv2d(horiz_img, k_sobel_y, padding=1).squeeze().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.5, 3.0))
ax1.imshow(horiz_np, cmap="gray", vmin=0, vmax=1)
ax1.set_title("Horizontal Lamellae")
ax1.axis("off")
ax2.imshow(edge_y, cmap="coolwarm")
ax2.set_title("Horizontal Sobel Response")
ax2.axis("off")
plt.tight_layout()
plt.show()

---
### 习题 7：推导并验证卷积 + 池化流水线的输出形状

In [ ]:
x_img = torch.zeros(4, 1, 32, 32)

# 手算推导:
# 1. Conv2d(1, 8, k=3, p=1):  (4, 8, 32, 32)
# 2. MaxPool2d(2, 2):         (4, 8, 16, 16)
# 3. Conv2d(8, 16, k=3, p=0): 16 - 3 + 1 = 14 -> (4, 16, 14, 14)
# 4. MaxPool2d(2, 2):         14 // 2 = 7     -> (4, 16, 7, 7)
feature_extractor = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(8, 16, kernel_size=3, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
)

out_feat = feature_extractor(x_img)
print("Output shape:", out_feat.shape, "(Expected: torch.Size([4, 16, 7, 7]))")

---
### 习题 8：训练轻量级 CNN 区分“水平层片”与“垂直层片”微结构

In [ ]:
rng = np.random.RandomState(42)
yy, xx = np.mgrid[0:24, 0:24].astype(np.float32) / 24.0
imgs_8, lbls_8 = [], []
for i in range(160):
    cls = i % 2
    freq = rng.uniform(3.5, 6.5)
    phase = rng.uniform(0, 2 * np.pi)
    coord = yy if cls == 0 else xx
    clean = (np.sin(2 * np.pi * freq * coord + phase) > 0).astype(np.float32) * 0.7 + 0.15
    noisy = np.clip(clean + rng.normal(0, 0.08, (24, 24)).astype(np.float32), 0.0, 1.0)
    imgs_8.append(noisy)
    lbls_8.append(cls)

X8 = torch.tensor(np.array(imgs_8)[:, None, :, :])
y8 = torch.tensor(lbls_8, dtype=torch.long)
X8_tr, X8_te = X8[:120], X8[120:]
y8_tr, y8_te = y8[:120], y8[120:]

torch.manual_seed(42)
mini_cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(8, 2),
)

loader_8 = DataLoader(TensorDataset(X8_tr, y8_tr), batch_size=24, shuffle=True)
opt_8 = torch.optim.Adam(mini_cnn.parameters(), lr=1e-2)
ce_8 = nn.CrossEntropyLoss()

for _ in range(25):
    mini_cnn.train()
    for xb, yb in loader_8:
        loss = ce_8(mini_cnn(xb), yb)
        opt_8.zero_grad()
        loss.backward()
        opt_8.step()

mini_cnn.eval()
with torch.no_grad():
    preds_8 = mini_cnn(X8_te).argmax(dim=1)
    acc_8 = (preds_8 == y8_te).float().mean().item()

print(f"Mini-CNN 参数总数: {sum(p.numel() for p in mini_cnn.parameters())}")
print(f"测试集分类准确率:  {acc_8 * 100:.1f}%")